# 2021 Census of Population: Census Profile

The census profile presents information from the 2021 Census of Population at dissemination areas level in Ontario province. On this version, the area of interest is the Great Golden Horseshoe.

## Reading a sample of the CSV file

This is a data subset reading from the 2.5gb size census 2021 csv file.

In [1]:
# import libraries
import pandas as pd
import duckdb

In [3]:
# show all columns
pd.set_option('display.max_columns', None)

The **Census Profile, 2021 Census of Population** is a large dataset. To identify the proper settings to import the full dataset. First, explore a sample. Previously, some basic `read_csv` configuration has been identify to make it work.
Also it helps to capture the `dtypes` and built a proper variables dictionary.

### Identify the encoding

In [5]:
# import libraries
import gzip
from charset_normalizer import from_bytes

In [6]:
# define the raw data path
path = "data/raw/da_c2021_dataset.csv.gz"

In [7]:
# open a sample
with gzip.open(path, "rb") as f:
    sample = f.read(5_000_000)

# analyzes bytes from the sample and selects the best character encoding candidate
result = from_bytes(sample).best()

print("Detected encoding:", result.encoding)
print("Chaos:", result.percent_chaos)


Detected encoding: cp1250
Chaos: 0.0


In [8]:
# try decoding with 'utf-8'
#  if it doesn't work find the bad byte character
try:
    sample.decode("utf-8")
    print("Sample is valid UTF-8")
except UnicodeDecodeError as e:
    print("Sample is NOT valid UTF-8")
    print("Bad byte:", sample[e.start:e.end])

Sample is NOT valid UTF-8
Bad byte: b'\xc9'


In [9]:
# try other encoding candidates
encodings = ["utf-8", "cp1250", "cp1252", "latin-1"]

for encoding in encodings:
    try:
        text = sample.decode(encoding)
        print(f"{encoding:>8}: ✓ valid")
    except UnicodeDecodeError:
        print(f"{encoding:>8}: ✗ invalid")

   utf-8: ✗ invalid
  cp1250: ✓ valid
  cp1252: ✓ valid
 latin-1: ✓ valid


In [10]:
# visually inspect the valid encoders
try:
    sample.decode("utf-8")
except UnicodeDecodeError as e:
    start = max(0, e.start - 100)
    end = e.end + 100

    for encoding in ["cp1250", "cp1252", "latin-1"]:
        print(f"\n--- {encoding} ---")
        print(sample[start:end].decode(encoding))


--- cp1250 ---
940,"",0,"",0,"",0,""
2021,"2021A000011124","01","Country","Canada",3.1,4.3,"20000",653,"          Éwé",,1955,"",1030,"",920,"",0,"",0,"",0,""
2021,"2021A000011124","01","Country","Canada",3.1,4.3,"2

--- cp1252 ---
940,"",0,"",0,"",0,""
2021,"2021A000011124","01","Country","Canada",3.1,4.3,"20000",653,"          Éwé",,1955,"",1030,"",920,"",0,"",0,"",0,""
2021,"2021A000011124","01","Country","Canada",3.1,4.3,"2

--- latin-1 ---
940,"",0,"",0,"",0,""
2021,"2021A000011124","01","Country","Canada",3.1,4.3,"20000",653,"          Éwé",,1955,"",1030,"",920,"",0,"",0,"",0,""
2021,"2021A000011124","01","Country","Canada",3.1,4.3,"2


In [11]:
# the odd-quote test
#  `rb` stands for read binary
with gzip.open(path, "rb") as f:
    for line_number, line in enumerate(f, 1):
        quotes = line.count(b'"')

        if quotes % 2 != 0:
            print("Odd quote count!")
            print("Physical line:", line_number)
            print("Quote count:", quotes)
            print(repr(line[:2000]))
            break
    else:
        print("No lines with odd quote counts")

No lines with odd quote counts


Three encoding works for the `da_c2021_dataset.csv.gz` datset: `cp1250`, `cp1252`, and `latin-1`. Based on the purpose of each encoding type, `cp1250` will be used as its intended to work in English, and French. Since Stats Canada releases data in both languagues this seems appropiate.

## Create a DuckDB database with the dataset

The file `da_c2021_dataset.csv.gz` has a ~765mb file size. Using pandas to process the whole dataset makes a `WSL` environment crash constantly. Instead, a DuckDB dataset will make it easy to make any process and transformation using SQL queries.

In [12]:
# Create a DuckDB database
raw_con = duckdb.connect("data/database/raw.duckdb")

### Exploring a sample

In [13]:
# detect csv config
df = raw_con.execute("""
    SELECT *
    FROM sniff_csv(
        ?,
        encoding='cp1252',
        delim=',',
        sample_size=20480
    )
""", ["data/raw/da_c2021_dataset.csv.gz"]).fetchdf()

print(df.T)

                                                                  0
Delimiter                                                         ,
Quote                                                             "
Escape                                                            "
NewLineDelimiter                                               \r\n
Comment                                                     (empty)
SkipRows                                                          0
HasHeader                                                      True
Columns           [{'name': 'CENSUS_YEAR', 'type': 'BIGINT'}, {'...
DateFormat                                                     None
TimestampFormat                                                None
UserArguments       delim=',', encoding='cp1252', sample_size=20480
Prompt            FROM read_csv('data/raw/da_c2021_dataset.csv.g...


In [14]:
# return the read_csv config 
prompt = df["Prompt"].iloc[0]
prompt = prompt[:-1]
prompt

'FROM read_csv(\'data/raw/da_c2021_dataset.csv.gz\', auto_detect=false, quote=\'"\', escape=\'"\', new_line=\'\\r\\n\', skip=0, comment=\'\', header=true, columns={\'CENSUS_YEAR\': \'BIGINT\', \'DGUID\': \'VARCHAR\', \'ALT_GEO_CODE\': \'VARCHAR\', \'GEO_LEVEL\': \'VARCHAR\', \'GEO_NAME\': \'VARCHAR\', \'TNR_SF\': \'DOUBLE\', \'TNR_LF\': \'DOUBLE\', \'DATA_QUALITY_FLAG\': \'VARCHAR\', \'CHARACTERISTIC_ID\': \'BIGINT\', \'CHARACTERISTIC_NAME\': \'VARCHAR\', \'CHARACTERISTIC_NOTE\': \'BIGINT\', \'C1_COUNT_TOTAL\': \'DOUBLE\', \'SYMBOL\': \'VARCHAR\', \'C2_COUNT_MEN+\': \'DOUBLE\', \'SYMBOL_1\': \'VARCHAR\', \'C3_COUNT_WOMEN+\': \'DOUBLE\', \'SYMBOL_2\': \'VARCHAR\', \'C10_RATE_TOTAL\': \'DOUBLE\', \'SYMBOL_3\': \'VARCHAR\', \'C11_RATE_MEN+\': \'DOUBLE\', \'SYMBOL_4\': \'VARCHAR\', \'C12_RATE_WOMEN+\': \'DOUBLE\', \'SYMBOL_5\': \'VARCHAR\'}, delim=\',\', encoding=\'cp1252\', sample_size=20480)'

In [15]:
# test config
raw_con.execute(f"""
    SELECT *
    {prompt}
    LIMIT 100
""").fetchdf()


,CENSUS_YEAR,DGUID,ALT_GEO_CODE,GEO_LEVEL,GEO_NAME,TNR_SF,TNR_LF,DATA_QUALITY_FLAG,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_NOTE,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2,C10_RATE_TOTAL,SYMBOL_3,C11_RATE_MEN+,SYMBOL_4,C12_RATE_WOMEN+,SYMBOL_5
0,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,1,"Population, 2021",1,36991981.0,None,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
1,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,2,"Population, 2016",1,35151728.0,None,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
2,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,3,"Population percentage change, 2016 to 2021",<NA>,5.2,None,NaN,...,NaN,...,5.2,NaN,NaN,...,NaN,...
3,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,4,Total private dwellings,2,16284235.0,None,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
4,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,5,Private dwellings occupied by usual residents,3,14978941.0,None,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,96,Total - Persons not in census families in pr...,<NA>,6850005.0,None,3320385.0,NaN,3529620.0,NaN,18.9,NaN,18.5,NaN,19.2,NaN
96,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,97,Living alone,<NA>,4396015.0,None,2048495.0,NaN,2347520.0,NaN,12.1,NaN,11.4,NaN,12.8,NaN
97,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,98,Living with other relatives,7,927345.0,None,393085.0,NaN,534260.0,NaN,2.6,NaN,2.2,NaN,2.9,NaN
98,2021,2021A000011124,01,Country,Canada,3.1,4.3,20000,99,Living with non-relatives only,<NA>,1526645.0,None,878800.0,NaN,647840.0,NaN,4.2,NaN,4.9,NaN,3.5,NaN


In [16]:
# describe column config
raw_con.execute(f"""
    DESCRIBE SELECT *
    {prompt}
""").fetchdf()

,column_name,column_type,null,key,default,extra
0,CENSUS_YEAR,BIGINT,YES,None,None,None
1,DGUID,VARCHAR,YES,None,None,None
2,ALT_GEO_CODE,VARCHAR,YES,None,None,None
3,GEO_LEVEL,VARCHAR,YES,None,None,None
4,GEO_NAME,VARCHAR,YES,None,None,None
5,TNR_SF,DOUBLE,YES,None,None,None
6,TNR_LF,DOUBLE,YES,None,None,None
7,DATA_QUALITY_FLAG,VARCHAR,YES,None,None,None
8,CHARACTERISTIC_ID,BIGINT,YES,None,None,None
9,CHARACTERISTIC_NAME,VARCHAR,YES,None,None,None


## Data dictionary

In order to built a proper variable dictionary, information has been pulled from the *metadata file* included in the download zip file. Geographic information definitions has been gathered from the [Full Table Download (CSV) User Guide](https://www.statcan.gc.ca/en/developers/csv/user-guide). Meaning that a full variable dictionary doesn't exist as such, although the information is available but scattered.

Variable            | Description | Dtype
--------------------|-------------|----------
CENSUS_YEAR         | year of census | BIGINT  
DGUID               | Dissemination Geography Unique Identifier - DGUID. <br>It is an alphanumeric code, composed of four components.<br>It varies from 10 to 20 characters in length. <br>The first 9 characters are fixed in composition and length.<br>Vintage (4) + Type (1) + Schema (4) + Geographic Unique Identifier (2-11) :<br>VVVV T SSSS  GGGGGGGGGGG <br> Further information at the [Dictionary, Census of Population, 2021: Dissemination Geography Unique Identifier (DGUID)](https://www12.statcan.gc.ca/census-recensement/2021/ref/dict/az/definition-eng.cfm?ID=geo055) | VARCHAR
ALT_GEO_CODE        | is an alternate geographic code that is usually equal to the ending digits of the DGUID.<br> This code is often the geographic code found in previous census cycle products | VARCHAR  
GEO_LEVEL           | this is the level of the geography. There are five levels in hierarchical order: Country, Province, Census divsion,  Census Subdivision, Dissemination Area | VARCHAR 
GEO_NAME            | the name of the geographic area | VARCHAR 
TNR_SF              | this value would be the total non-response rate to the short-form questionnaire | DOUBLE
TNR_LF              | this value would be the total non-response rate to the long-form questionnaire (*) | DOUBLE
DATA_QUALITY_FLAG   | its a 5 character number that describes data quality | VARCHAR
CHARACTERISTIC_ID   | identifier of each one of the 2631 characteristics.<br> i.e.: *Total - Age groups of the population - 100% data*, *Total private dwelling*, etc. | BIGINT  
CHARACTERISTIC_NAME | is a descriptibe name for each characteristic associated with an identifier number. <br>Characteristics are organized by topic and subtopic. <br>More info at [Characteristics by topic and subtopic](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/about-apropos/metadata-metadonnees-eng.cfm) | VARCHAR
CHARACTERISTIC_NOTE | is a reference to the 204 footnotes mentioned in the previous table | BIGINT
C1_COUNT_TOTAL      | is the count of total population | DOUBLE
SYMBOL              | field used to associate any necessary quality symbols `C1_COUNT_TOTAL` according to the [Standard table symbols] also applicable to other fields with the pattern `SYMBOL_N`  (https://www.statcan.gc.ca/en/concepts/definitions/guide-symbol)| VARCHAR
C2_COUNT_MEN+       | includes men and boys, as well as some non-binary persons | DOUBLE
SYMBOL_1              | field used to associate any necessary quality symbols `C2_COUNT_MEN+` | VARCHAR
C3_COUNT_WOMEN+     | includes women and girls, as well as some non-binary persons | DOUBLE
SYMBOL_2              | field used to associate any necessary quality symbols `C3_COUNT_WOMEN+` | VARCHAR
C10_RATE_TOTAL      | corresponding rate for `C1_COUNT_TOTAL` | DOUBLE
SYMBOL_3              | field used to associate any necessary quality symbols `C10_RATE_TOTAL` | VARCHAR
C11_RATE_MEN+       | corresponding rate for `C2_COUNT_MEN+` | DOUBLE
SYMBOL_4              | field used to associate any necessary quality symbols `C11_RATE_MEN+` | VARCHAR
C12_RATE_WOMEN+     | corresponding rate for `C3_COUNT_WOMEN+` | DOUBLE
SYMBOL_5              | field used to associate any necessary quality symbols `C12_RATE_WOMEN+` | VARCHAR

## Improving quality of import

### Column data types

In [17]:
# inspect columns config
raw_con.execute(f"""
    DESCRIBE SELECT *
    {prompt}
""").fetchdf()

,column_name,column_type,null,key,default,extra
0,CENSUS_YEAR,BIGINT,YES,None,None,None
1,DGUID,VARCHAR,YES,None,None,None
2,ALT_GEO_CODE,VARCHAR,YES,None,None,None
3,GEO_LEVEL,VARCHAR,YES,None,None,None
4,GEO_NAME,VARCHAR,YES,None,None,None
5,TNR_SF,DOUBLE,YES,None,None,None
6,TNR_LF,DOUBLE,YES,None,None,None
7,DATA_QUALITY_FLAG,VARCHAR,YES,None,None,None
8,CHARACTERISTIC_ID,BIGINT,YES,None,None,None
9,CHARACTERISTIC_NAME,VARCHAR,YES,None,None,None


In [18]:
new_prompt = (
    'FROM read_csv('
        '\'data/raw/da_c2021_dataset.csv.gz\', '
        'auto_detect=false, '
        'quote=\'"\', '
        'escape=\'"\', '
        'new_line=\'\\r\\n\', '
        'skip=0, comment=\'\', '
        'header=true, '
        'columns={'
            '\'CENSUS_YEAR\': \'SMALLINT\', '  # this field was 'BIGINT'
            '\'DGUID\': \'VARCHAR\', '
            '\'ALT_GEO_CODE\': \'VARCHAR\', '
            '\'GEO_LEVEL\': \'VARCHAR\', '
            '\'GEO_NAME\': \'VARCHAR\', '
            '\'TNR_SF\': \'DOUBLE\', '
            '\'TNR_LF\': \'DOUBLE\', '
            '\'DATA_QUALITY_FLAG\': \'VARCHAR\', '
            '\'CHARACTERISTIC_ID\': \'SMALLINT\', ' # this field was 'BIGINT'
            '\'CHARACTERISTIC_NAME\': \'VARCHAR\', '
            '\'CHARACTERISTIC_NOTE\': \'SMALLINT\', ' # this field was 'BIGINT'
            '\'C1_COUNT_TOTAL\': \'DOUBLE\', '
            '\'SYMBOL\': \'VARCHAR\', '
            '\'C2_COUNT_MEN+\': \'DOUBLE\', '
            '\'SYMBOL_1\': \'VARCHAR\', '
            '\'C3_COUNT_WOMEN+\': \'DOUBLE\', '
            '\'SYMBOL_2\': \'VARCHAR\', '
            '\'C10_RATE_TOTAL\': \'DOUBLE\', '
            '\'SYMBOL_3\': \'VARCHAR\', '
            '\'C11_RATE_MEN+\': \'DOUBLE\', '
            '\'SYMBOL_4\': \'VARCHAR\', '
            '\'C12_RATE_WOMEN+\': \'DOUBLE\', '
            '\'SYMBOL_5\': \'VARCHAR\'}, '
        'delim=\',\', '
        'encoding=\'cp1252\', '
        'sample_size=20480)'
)

In [19]:
# inspect new columns config
raw_con.execute(f"""
    DESCRIBE SELECT *
    {new_prompt}
""").fetchdf()

,column_name,column_type,null,key,default,extra
0,CENSUS_YEAR,SMALLINT,YES,None,None,None
1,DGUID,VARCHAR,YES,None,None,None
2,ALT_GEO_CODE,VARCHAR,YES,None,None,None
3,GEO_LEVEL,VARCHAR,YES,None,None,None
4,GEO_NAME,VARCHAR,YES,None,None,None
5,TNR_SF,DOUBLE,YES,None,None,None
6,TNR_LF,DOUBLE,YES,None,None,None
7,DATA_QUALITY_FLAG,VARCHAR,YES,None,None,None
8,CHARACTERISTIC_ID,SMALLINT,YES,None,None,None
9,CHARACTERISTIC_NAME,VARCHAR,YES,None,None,None


## Filtering Census by city, geographical level and variables of interest

### By Dissemination Geography Unique Identifier `DGUID` (`ALT_GEO_CODE` by proxy)

The Great Golden Horseshoe is the area of interest for this project. The complementary csv file `Geo_starting_now.csv` has a list of all `DGUID`, listed as `Geo Code` in the dataset, along with their corresponding `Geo Names` and the starting line number of each geographical unit. The `Line Number` variable can be useful for improving the `read_csv` configuration by defining the start and end lines for import and enabling early-stage filtering.

Geo Code | Geo Name | Line Number
---------|----------|------------
2021S0503535 | Toronto | 10816043
2021S05075350001.00 | 5350001 | 10818674

In a GIS environment, the Administrative boundaries in the area of interest has been explored. As a result, it has been identified the Census Division `DGUID`'s that contain the dissemination areas that need to be included on this research. The variable `ALT_GEO_CODE` will be useful to filter the Ontario dataset.

In [20]:
# filter for 'ALT_GEO_CODE' field
alt_geo_code_list = (
    "'3514','3515','3516','3518','3519',"
    "'3520','3521','3522','3523','3524',"
    "'3525','3526','3527','3528','3529',"
    "'3530','3543'"
)
alt_geo_code_list

"'3514','3515','3516','3518','3519','3520','3521','3522','3523','3524','3525','3526','3527','3528','3529','3530','3543'"

### By geographical level `GEO_LEVEL`
The dataset includes information at multiple geographical levels listed on the `GEO_LEVEL` field. There are five levels in order or hierarchy: Country, Province, Census divsion,  Census Subdivision, Dissemination. This project will focus exclusiely on the `Dissemination Area` level. 

The `Geo Code` structure varies depending on the geographical level. According to the [Dictionary, Census of Population, 2021: Dissemination Area (DA))](https://www12.statcan.gc.ca/census-recensement/2021/ref/dict/az/definition-eng.cfm?ID=geo021), the coding structure for DAs follows a specific naming convention:

>Each dissemination area (DA) is assigned a four-digit code. In order to uniquely identify each DA in Canada, the two-digit province/territory (PR) code and the two-digit census division (CD) code must precede the DA code. For example:

PR-CD-DA code	| Description
----------------|------------
12 09 0103	| Province 12: Nova Scotia<br> CD 09: Halifax<br> DA 0103
59 09 0103	| Province 59: British Columbia<br> CD 09: Fraser Valley<br> DA 0103

### Create a raw database

In [21]:
# create raw table
raw_con.execute(f"""
    CREATE OR REPLACE TABLE raw_da_c2021 AS
    SELECT *
    {new_prompt}
""")

In [22]:
# test filters
raw_con.execute(f"""
    SELECT
        *
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    LIMIT 100
""").fetch_df()

,CENSUS_YEAR,DGUID,ALT_GEO_CODE,GEO_LEVEL,GEO_NAME,TNR_SF,TNR_LF,DATA_QUALITY_FLAG,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_NOTE,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2,C10_RATE_TOTAL,SYMBOL_3,C11_RATE_MEN+,SYMBOL_4,C12_RATE_WOMEN+,SYMBOL_5
0,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,1,"Population, 2021",1,503.0,NaN,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
1,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,2,"Population, 2016",1,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
2,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,3,"Population percentage change, 2016 to 2021",<NA>,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
3,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,4,Total private dwellings,2,193.0,NaN,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
4,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,5,Private dwellings occupied by usual residents,3,186.0,NaN,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,96,Total - Persons not in census families in pr...,<NA>,55.0,NaN,35.0,NaN,20.0,NaN,11.0,NaN,13.5,NaN,8.5,NaN
96,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,97,Living alone,<NA>,30.0,NaN,20.0,NaN,10.0,NaN,6.0,NaN,7.7,NaN,4.3,NaN
97,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,98,Living with other relatives,7,10.0,NaN,5.0,NaN,0.0,NaN,2.0,NaN,1.9,NaN,0.0,NaN
98,2021,2021S051235140184,35140184,Dissemination area,35140184,0.5,2.1,00000,99,Living with non-relatives only,<NA>,15.0,NaN,10.0,NaN,5.0,NaN,3.0,NaN,3.8,NaN,2.1,NaN


### Inspect char errors in Characteristic Name

In [23]:
# return distinct characteristic name
raw_con.execute(f"""
    SELECT 
        DISTINCT CHARACTERISTIC_NAME
    FROM
        raw_da_c2021
    ORDER BY CHARACTERISTIC_ID
    LIMIT 2631*1
""").fetchdf()

,CHARACTERISTIC_NAME
0,"Population, 2021"
1,"Population, 2016"
2,"Population percentage change, 2016 to 2021"
3,Total private dwellings
4,Private dwellings occupied by usual residents
...,...
1757,Children eligible for instruction in the min...
1758,Children not eligible for instruction in the...
1759,Total - Eligibility and instruction in the min...
1760,Eligible children who have been instructed...


### Handling constant variables

The `CENSUS_YEAR` and `GEO_LEVEL` are now constant variables, so there is no reason to keep it as a repeated value in the dataframe. 

These variables consumes resources unnecesarily, and doesn't add up to further analysis. For that reason, its values will be registed here as a text string and filter out from the database.

In [24]:
# drop constant fields
raw_con.execute(f"""
    SELECT
        DGUID, ALT_GEO_CODE, 
        GEO_NAME,
        TNR_SF, TNR_LF, DATA_QUALITY_FLAG,
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        CHARACTERISTIC_NOTE,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2, 
        C10_RATE_TOTAL, SYMBOL_3, 
        "C11_RATE_MEN+", SYMBOL_4,
        "C12_RATE_WOMEN+", SYMBOL_5 
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    LIMIT 100
""").fetch_df()

,DGUID,ALT_GEO_CODE,GEO_NAME,TNR_SF,TNR_LF,DATA_QUALITY_FLAG,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_NOTE,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2,C10_RATE_TOTAL,SYMBOL_3,C11_RATE_MEN+,SYMBOL_4,C12_RATE_WOMEN+,SYMBOL_5
0,2021S051235140184,35140184,35140184,0.5,2.1,00000,1,"Population, 2021",1,503.0,NaN,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
1,2021S051235140184,35140184,35140184,0.5,2.1,00000,2,"Population, 2016",1,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
2,2021S051235140184,35140184,35140184,0.5,2.1,00000,3,"Population percentage change, 2016 to 2021",<NA>,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
3,2021S051235140184,35140184,35140184,0.5,2.1,00000,4,Total private dwellings,2,193.0,NaN,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
4,2021S051235140184,35140184,35140184,0.5,2.1,00000,5,Private dwellings occupied by usual residents,3,186.0,NaN,NaN,...,NaN,...,NaN,...,NaN,...,NaN,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2021S051235140184,35140184,35140184,0.5,2.1,00000,96,Total - Persons not in census families in pr...,<NA>,55.0,NaN,35.0,NaN,20.0,NaN,11.0,NaN,13.5,NaN,8.5,NaN
96,2021S051235140184,35140184,35140184,0.5,2.1,00000,97,Living alone,<NA>,30.0,NaN,20.0,NaN,10.0,NaN,6.0,NaN,7.7,NaN,4.3,NaN
97,2021S051235140184,35140184,35140184,0.5,2.1,00000,98,Living with other relatives,7,10.0,NaN,5.0,NaN,0.0,NaN,2.0,NaN,1.9,NaN,0.0,NaN
98,2021S051235140184,35140184,35140184,0.5,2.1,00000,99,Living with non-relatives only,<NA>,15.0,NaN,10.0,NaN,5.0,NaN,3.0,NaN,3.8,NaN,2.1,NaN


### Filter out other variables

Other variables are been removed since visual inspection makes the redundancy of values evident. Such as in `DGUID`, `ALT_GEO_CODE`, `GEO_NAME`.

All of these variables have the same purpose and `DGUID` will be only preserved,  as it returns the full `Dissemination Geographic Unique ID` value.

Following the data dictionary, the variables `TNR_SF`, `TNR_LF` are not relevant for this analysis since they return non-response rates. 

The variables rates variables `C10_RATE_TOTAL`, `C11_RATE_MEN+`, `C12_RATE_WOMEN+` since they return rates values that are not relevant for this analysis. Also, the fields quality associated fields are being removed.

In [25]:
# drop non-relevant fields
raw_con.execute(f"""
    SELECT
        DGUID, 
        DATA_QUALITY_FLAG,
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        CHARACTERISTIC_NOTE,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    LIMIT 100
""").fetch_df()

,DGUID,DATA_QUALITY_FLAG,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_NOTE,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,00000,1,"Population, 2021",1,503.0,NaN,NaN,...,NaN,...
1,2021S051235140184,00000,2,"Population, 2016",1,NaN,...,NaN,...,NaN,...
2,2021S051235140184,00000,3,"Population percentage change, 2016 to 2021",<NA>,NaN,...,NaN,...,NaN,...
3,2021S051235140184,00000,4,Total private dwellings,2,193.0,NaN,NaN,...,NaN,...
4,2021S051235140184,00000,5,Private dwellings occupied by usual residents,3,186.0,NaN,NaN,...,NaN,...
...,...,...,...,...,...,...,...,...,...,...,...
95,2021S051235140184,00000,96,Total - Persons not in census families in pr...,<NA>,55.0,NaN,35.0,NaN,20.0,NaN
96,2021S051235140184,00000,97,Living alone,<NA>,30.0,NaN,20.0,NaN,10.0,NaN
97,2021S051235140184,00000,98,Living with other relatives,7,10.0,NaN,5.0,NaN,0.0,NaN
98,2021S051235140184,00000,99,Living with non-relatives only,<NA>,15.0,NaN,10.0,NaN,5.0,NaN


## Data quality indicators

The variable `DATA_QUALITY_FLAG` describes 5 quality markers with flags. The details of each marker and the meaning behind each flag is available at [2021 Census data quality indicators](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/about-apropos/about-apropos.cfm?Lang=E#dq-qd). What matters for the EDA is to identify if there is a one-to-one relationship between the `DATA_QUALITY_FLAG` and the geographical units.

In [26]:
# drop non-relevant fields
raw_con.execute(f"""
    SELECT
        COUNT(DISTINCT DATA_QUALITY_FLAG)
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    LIMIT 100
""").fetchone()[0]

40

There are **40** quality flags listed in the AOI census tracts.

Validating if there is only `DATA_QUALITY_FLAG` per geographical unit.

In [27]:
# Unique DATA_QUALITY_FLAG values
raw_con.execute(f"""
    SELECT
        DGUID,
        COUNT(DISTINCT DATA_QUALITY_FLAG) AS n_flags
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    GROUP BY
        DGUID
    ORDER BY n_flags DESC
""").fetch_df()["n_flags"].unique().item()

1

Each `DGUID` has a unique `DATA_QUALITY_FLAG` value, and the data quality markers are consistent across geographical units and characteristics.

This validates that there is a one-to-one relationship between `DGUID` and `DATA_QUALITY_FLAG`. This means that it is possible to store the quality indicators in a complemetary table, along with one of the geo codes. This operations will help making the main dataframe more efficient and less redundant.

In [28]:
# store `DGUID` and `DATA_QUALITY_FLAG` in another table
raw_con.execute(f"""
    CREATE OR REPLACE TABLE 
        raw_da_c2021_data_quality_flag AS
    SELECT
        DGUID,
        COUNT(DISTINCT DATA_QUALITY_FLAG) AS n_flags
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    GROUP BY
        DGUID
    ORDER BY n_flags DESC
""")

In [29]:
raw_con.execute(f"""
    SELECT *
    FROM 
        raw_da_c2021_data_quality_flag
    LIMIT 100
""").fetch_df()

,DGUID,n_flags
0,2021S051235260285,1
1,2021S051235300730,1
2,2021S051235211474,1
3,2021S051235212299,1
4,2021S051235211638,1
...,...,...
95,2021S051235180733,1
96,2021S051235200413,1
97,2021S051235201417,1
98,2021S051235202937,1


In [30]:
# drop non-relevant fields
raw_con.execute(f"""
    SELECT
        DGUID, 
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        CHARACTERISTIC_NOTE,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    LIMIT 100
""").fetch_df()

,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_NOTE,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,1,"Population, 2021",1,503.0,NaN,NaN,...,NaN,...
1,2021S051235140184,2,"Population, 2016",1,NaN,...,NaN,...,NaN,...
2,2021S051235140184,3,"Population percentage change, 2016 to 2021",<NA>,NaN,...,NaN,...,NaN,...
3,2021S051235140184,4,Total private dwellings,2,193.0,NaN,NaN,...,NaN,...
4,2021S051235140184,5,Private dwellings occupied by usual residents,3,186.0,NaN,NaN,...,NaN,...
...,...,...,...,...,...,...,...,...,...,...
95,2021S051235140184,96,Total - Persons not in census families in pr...,<NA>,55.0,NaN,35.0,NaN,20.0,NaN
96,2021S051235140184,97,Living alone,<NA>,30.0,NaN,20.0,NaN,10.0,NaN
97,2021S051235140184,98,Living with other relatives,7,10.0,NaN,5.0,NaN,0.0,NaN
98,2021S051235140184,99,Living with non-relatives only,<NA>,15.0,NaN,10.0,NaN,5.0,NaN


## Characteristic ID, name and note

According to the dictionary there is a unique id per characteristic, and a note per characteristic.

In [31]:
# Count unique CHARACTERISTIC_ID
raw_con.execute(f"""
    SELECT
        COUNT (DISTINCT CHARACTERISTIC_ID)
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
""").fetchone()[0]

2631

There are **2631** unique characteristc ids, which is consistent with expected quantity of variables in the census.

In [32]:
# Count unique CHARACTERISTIC_NAME
raw_con.execute(f"""
    SELECT 
        COUNT (DISTINCT CHARACTERISTIC_NAME)
    FROM
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
""").fetchone()[0]

1762

There are **1762** unique characteristic names, which indicates that some names are repeated across different characteristics. A quick review of the [Census Profile metadata: Characteristics by topic and subtopic](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/about-apropos/metadata-metadonnees-eng.cfm) shows that the naming of characteristics is hierarchical, as some characteristics aggregate quantities from others.

In [33]:
# For example
raw_con.execute(f"""
    SELECT DISTINCT 
        CHARACTERISTIC_ID,
        CHARACTERISTIC_NAME
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    ORDER BY
        CHARACTERISTIC_ID
""").fetch_df().iloc[33:38]

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME
33,34,Total - Distribution (%) of the population by ...
34,35,0 to 14 years
35,36,15 to 64 years
36,37,65 years and over
37,38,85 years and over


Knowing that there are aggratated characteristics will help with better analysis of the variables of interest.

In [34]:
# Number of CHARACTERISTIC_NOTE per CHARACTERISTIC_ID
#  the varchar field `CHARACTERISTIC_NOTE` has NULL values
#  so NULLs has to be replaced with 0 to be properly counted
raw_con.execute(f"""
    SELECT 
        CHARACTERISTIC_ID,
        COUNT(DISTINCT COALESCE(CHARACTERISTIC_NOTE, '0')) AS n_notes
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    GROUP BY
        CHARACTERISTIC_ID
    ORDER BY 
        CHARACTERISTIC_ID
""").fetch_df()["n_notes"].unique().item()

1

Each `CHARACTERISTIC_ID` has a unique `CHARACTERISTIC_NOTE` value, and the notes are consistent across geographical units and characteristics.

The characteristcs variables will be stored in a complementary table to improve efficiency. The main table will only keep the `CHARACTERISTIC_NAME` column after splitting the dataset by subtopic and characteristic.

In [35]:
# store `CHARACTERISTIC_ID`, `CHARACTERISTIC_NAME` 
#  and `CHARACTERISTIC_NOTE` in another table
raw_con.execute(f"""
    CREATE OR REPLACE TABLE
        raw_da_c2021_characteristic_note AS
    SELECT DISTINCT
        CHARACTERISTIC_ID, 
        CHARACTERISTIC_NAME,
        COALESCE(CHARACTERISTIC_NOTE, '0') AS CHARACTERISTIC_NOTE
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    ORDER BY
        CHARACTERISTIC_ID
""")

In [36]:
raw_con.execute(f"""
    SELECT *
    FROM 
        raw_da_c2021_characteristic_note
    LIMIT 100
""").fetch_df()

,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,CHARACTERISTIC_NOTE
0,1,"Population, 2021",1
1,2,"Population, 2016",1
2,3,"Population percentage change, 2016 to 2021",0
3,4,Total private dwellings,2
4,5,Private dwellings occupied by usual residents,3
...,...,...,...
95,96,Total - Persons not in census families in pr...,0
96,97,Living alone,0
97,98,Living with other relatives,7
98,99,Living with non-relatives only,0


In [37]:
# drop non-relevant fields
raw_con.execute(f"""
    SELECT
        DGUID, 
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
    LIMIT 100
""").fetch_df()

,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,1,"Population, 2021",503.0,NaN,NaN,...,NaN,...
1,2021S051235140184,2,"Population, 2016",NaN,...,NaN,...,NaN,...
2,2021S051235140184,3,"Population percentage change, 2016 to 2021",NaN,...,NaN,...,NaN,...
3,2021S051235140184,4,Total private dwellings,193.0,NaN,NaN,...,NaN,...
4,2021S051235140184,5,Private dwellings occupied by usual residents,186.0,NaN,NaN,...,NaN,...
...,...,...,...,...,...,...,...,...,...
95,2021S051235140184,96,Total - Persons not in census families in pr...,55.0,NaN,35.0,NaN,20.0,NaN
96,2021S051235140184,97,Living alone,30.0,NaN,20.0,NaN,10.0,NaN
97,2021S051235140184,98,Living with other relatives,10.0,NaN,5.0,NaN,0.0,NaN
98,2021S051235140184,99,Living with non-relatives only,15.0,NaN,10.0,NaN,5.0,NaN


## Replace table with filters

In [38]:
# replace raw table with new filters
raw_con.execute(f"""
    CREATE OR REPLACE 
        TABLE raw_da_c2021 AS
    SELECT
        DGUID, 
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        GEO_LEVEL = 'Dissemination area' AND
        LEFT(ALT_GEO_CODE, 4) IN ({alt_geo_code_list})
""")

In [39]:
# inspect
raw_con.execute(f"""
    SELECT
        *
    FROM 
        raw_da_c2021
    LIMIT 100
""").fetch_df()

,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,1,"Population, 2021",503.0,NaN,NaN,...,NaN,...
1,2021S051235140184,2,"Population, 2016",NaN,...,NaN,...,NaN,...
2,2021S051235140184,3,"Population percentage change, 2016 to 2021",NaN,...,NaN,...,NaN,...
3,2021S051235140184,4,Total private dwellings,193.0,NaN,NaN,...,NaN,...
4,2021S051235140184,5,Private dwellings occupied by usual residents,186.0,NaN,NaN,...,NaN,...
...,...,...,...,...,...,...,...,...,...
95,2021S051235140184,96,Total - Persons not in census families in pr...,55.0,NaN,35.0,NaN,20.0,NaN
96,2021S051235140184,97,Living alone,30.0,NaN,20.0,NaN,10.0,NaN
97,2021S051235140184,98,Living with other relatives,10.0,NaN,5.0,NaN,0.0,NaN
98,2021S051235140184,99,Living with non-relatives only,15.0,NaN,10.0,NaN,5.0,NaN


In [46]:
# return all table
raw_con.execute("""
    SELECT
        table_schema,
        table_name
    FROM 
        information_schema.tables
    WHERE 
        table_type = 'BASE TABLE';
""").fetch_df()


,table_schema,table_name
0,main,raw_da_c2021
1,main,raw_da_c2021_characteristic_note
2,main,raw_da_c2021_data_quality_flag


### Optimize Database file size

In [4]:
# import library
import os

In [5]:
# define path variables
raw = 'data/database/raw.duckdb'
compact ='data/database/raw_compact.duckdb'

In [6]:
# making sure that compact file is new
if os.path.exists(compact):
    os.remove(compact)

In [7]:
raw_con = duckdb.connect("data/database/raw.duckdb")

In [8]:
# create a fresh database (smaller file size)
raw_con.execute(f"""
    ATTACH 
        'data/database/raw_compact.duckdb' AS new;
    COPY 
        FROM DATABASE raw TO new;
    DETACH 
        new;
""")

In [9]:
# close connection
raw_con.close()

In [10]:
# replace original `raw` file with `raw_compact` file
os.remove(raw)
os.rename(compact, raw)

In [11]:
# reconnect
raw_con = duckdb.connect("data/database/raw.duckdb")

## Analysis by variables of interest

The variables of interest belong to the topic *Ethnocultural and religious diversity*, within this topic the subtopic of interest is *Ethnic or cultural origin*. Additionally, from the topic *Immigration, place of birth, and citizenship*, the subtopic *Selected places of birth for the immigrant population* will also be included among the selected variables.

The goal of the EDA is to identify which *characterstics* will be selected for further analysis. For now, all characteristics within the listed subtopics will be analyzed.

A quick search in the *General Information* txt file shows that the characteristics of interests and their id numbers are:

Characteristic | ID range
---------------|----------
Place of birth for the immigrant population in private households | 1544-1603
Place of birth for the recent immigrant population in private households | 1604-1664
Ethnic or cultural origin for the population in private households | 1698-1948

In [3]:
# filtering the characterists by subtopic
#  group 1: Place of birth for the immigrant population in private households
raw_con.execute(f"""
    SELECT
        DGUID, 
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1544 AND 1603
    LIMIT 100
""").fetch_df()

,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,1544,Total - Place of birth for the immigrant popul...,30.0,None,10.0,None,15.0,None
1,2021S051235140184,1545,Americas,0.0,None,0.0,None,0.0,None
2,2021S051235140184,1546,Brazil,0.0,None,0.0,None,0.0,None
3,2021S051235140184,1547,Colombia,0.0,None,0.0,None,0.0,None
4,2021S051235140184,1548,El Salvador,0.0,None,0.0,None,0.0,None
...,...,...,...,...,...,...,...,...,...
95,2021S051235140185,1579,Ethiopia,0.0,None,0.0,None,0.0,None
96,2021S051235140185,1580,Morocco,0.0,None,0.0,None,0.0,None
97,2021S051235140185,1581,Nigeria,0.0,None,0.0,None,0.0,None
98,2021S051235140185,1582,Somalia,0.0,None,0.0,None,0.0,None


In [4]:
# filtering the characterists by subtopic
#  group 2: Place of birth for the recent immigrant population in private households
raw_con.execute(f"""
    SELECT
        DGUID, 
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1604 AND 1664
    LIMIT 100
""").fetch_df()

,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,1604,Total - Place of birth for the recent immigran...,0.0,None,0.0,None,0.0,None
1,2021S051235140184,1605,Americas,0.0,None,0.0,None,0.0,None
2,2021S051235140184,1606,Brazil,0.0,None,0.0,None,0.0,None
3,2021S051235140184,1607,Colombia,0.0,None,0.0,None,0.0,None
4,2021S051235140184,1608,Haiti,0.0,None,0.0,None,0.0,None
...,...,...,...,...,...,...,...,...,...
95,2021S051235140185,1638,Other places of birth in Africa,0.0,None,0.0,None,0.0,None
96,2021S051235140185,1639,Asia,0.0,None,0.0,None,0.0,None
97,2021S051235140185,1640,Afghanistan,0.0,None,0.0,None,0.0,None
98,2021S051235140185,1641,Bangladesh,0.0,None,0.0,None,0.0,None


In [5]:
# filtering the characterists by subtopic
#  group 3: Ethnic or cultural origin for the population in private households
raw_con.execute(f"""
    SELECT
        DGUID, 
        CHARACTERISTIC_ID, CHARACTERISTIC_NAME,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1698 AND 1948
    LIMIT 100
""").fetch_df()

,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,1698,Total - Ethnic or cultural origin for the popu...,500.0,None,270.0,None,235.0,None
1,2021S051235140184,1699,Canadian,115.0,None,55.0,None,50.0,None
2,2021S051235140184,1700,English,160.0,None,75.0,None,85.0,None
3,2021S051235140184,1701,Irish,115.0,None,45.0,None,70.0,None
4,2021S051235140184,1702,Scottish,115.0,None,60.0,None,50.0,None
...,...,...,...,...,...,...,...,...,...
95,2021S051235140184,1793,"East or Southeast Asian, n.o.s.",0.0,None,0.0,None,0.0,None
96,2021S051235140184,1794,"West or Central Asian or Middle Eastern, n.o.s.",0.0,None,0.0,None,0.0,None
97,2021S051235140184,1795,"Caribbean, n.o.s.",0.0,None,0.0,None,0.0,None
98,2021S051235140184,1796,Algonquin,0.0,None,0.0,None,0.0,None


## Ethnic or cultural origin for the population in private households

According to the dictionary, *ethnic or cultural origin* refers to the ethnic or cultural background of a person's ancestors. These origins may be Indigenous origins, linked to specific countries, or other origins that are not tied to any partiular country.

In the *2021 Census of Population*, this information was  captured using a 25% sample, meaning that only one in four households received the long-form questionnaire, which included this question. The data is reported for individuals living in *private households*.

More information in [Ethnic or Cultural Origin Reference Guide, Census of Population, 2021](https://www12.statcan.gc.ca/census-recensement/2021/ref/98-500/008/98-500-x2021008-eng.cfm)

In [6]:
# filtering the characterists by subtopic
#  group 3: Ethnic or cultural origin for the population in private households
raw_con.execute(f"""
    SELECT
        DGUID, 
        CHARACTERISTIC_NAME,
        C1_COUNT_TOTAL, SYMBOL, 
        "C2_COUNT_MEN+", SYMBOL_1, 
        "C3_COUNT_WOMEN+", SYMBOL_2
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1698 AND 1948
    LIMIT 100
""").fetch_df()

,DGUID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140184,Total - Ethnic or cultural origin for the popu...,500.0,None,270.0,None,235.0,None
1,2021S051235140184,Canadian,115.0,None,55.0,None,50.0,None
2,2021S051235140184,English,160.0,None,75.0,None,85.0,None
3,2021S051235140184,Irish,115.0,None,45.0,None,70.0,None
4,2021S051235140184,Scottish,115.0,None,60.0,None,50.0,None
...,...,...,...,...,...,...,...,...
95,2021S051235140184,"East or Southeast Asian, n.o.s.",0.0,None,0.0,None,0.0,None
96,2021S051235140184,"West or Central Asian or Middle Eastern, n.o.s.",0.0,None,0.0,None,0.0,None
97,2021S051235140184,"Caribbean, n.o.s.",0.0,None,0.0,None,0.0,None
98,2021S051235140184,Algonquin,0.0,None,0.0,None,0.0,None


In [7]:
# number of characteristics per dissemination area
raw_con.execute(f"""
    SELECT
        DGUID, 
        COUNT (CHARACTERISTIC_NAME) AS n_char
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1698 AND 1948
    GROUP BY DGUID
""").fetch_df()["n_char"].unique().item()

251

There are **251** characteristics per dissemination area, some of them are aggregated in broader categories. There is a multi-level hierarchy that goes from total, continent, country, and more specific origin categories.

In [8]:
# get all unique Characteristic Name
raw_con.execute(f"""
    SELECT 
        DISTINCT CHARACTERISTIC_NAME
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1698 AND 1948
    ORDER BY
        DGUID, CHARACTERISTIC_ID
""").fetch_df().iloc[:, 0].tolist()

['Total - Ethnic or cultural origin for the population in private households - 25% sample data',
 '  Canadian',
 '  English',
 '  Irish',
 '  Scottish',
 '  French, n.o.s.',
 '  German',
 '  Chinese',
 '  Italian',
 '  Indian (India)',
 '  Ukrainian',
 '  Dutch',
 '  Polish',
 '  Québécois',
 '  British Isles, n.o.s.',
 '  Filipino',
 '  French Canadian',
 '  Caucasian (White), n.o.s.',
 '  First Nations (North American Indian), n.o.s.',
 '  Métis',
 '  European, n.o.s.',
 '  Russian',
 '  Norwegian',
 '  Welsh',
 '  Portuguese',
 '  American',
 '  Spanish',
 '  Swedish',
 '  Hungarian',
 '  Acadian',
 '  Pakistani',
 '  African, n.o.s.',
 '  Jewish',
 '  Punjabi',
 '  Vietnamese',
 '  Arab, n.o.s.',
 '  Greek',
 '  Jamaican',
 '  Asian, n.o.s.',
 '  Cree, n.o.s.',
 '  Korean',
 '  Romanian',
 '  Lebanese',
 '  Iranian',
 '  Christian, n.i.e.',
 '  Danish',
 '  North American Indigenous, n.o.s.',
 '  Sikh',
 '  Austrian',
 '  Belgian',
 '  Haitian',
 '  Hindu',
 '  Mexican',
 '  Mennon

According to the [List of ethnic or cultural origins 2021](https://www23.statcan.gc.ca/imdb/p3VD.pl?Function=getVD&TVD=1310929), the ancestry is aggregated in broader categories when possible. Such as in *European/British Isles/Irish*. This hierarchy is not represented in the `CHARACTERISTIC_NAME` variable. Further analysis may be needed, grouping the origins as listed.

In [9]:
# create a characteristic hierarchy column by 
#  capturing the white spaces before the string
raw_con.execute(f"""
    SELECT DISTINCT
        CHARACTERISTIC_NAME,
        (((
         LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
        ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1698 AND 1948
    ORDER BY
        DGUID, CHARACTERISTIC_ID
""").fetch_df()

,CHARACTERISTIC_NAME,CHARACTERISTIC_LEVEL
0,Total - Ethnic or cultural origin for the popu...,1
1,Canadian,2
2,English,2
3,Irish,2
4,Scottish,2
...,...,...
246,Karen,2
247,Edo,2
248,Tajik,2
249,Amhara,2


In [10]:
# map unique CHARACTERISTIC_LEVEL 
raw_con.execute(f"""
    SELECT DISTINCT
        CHARACTERISTIC_NAME,
        (((
         LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
        ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1698 AND 1948
    ORDER BY
        DGUID, CHARACTERISTIC_ID
""").fetch_df()["CHARACTERISTIC_LEVEL"].unique()

array([1, 2], dtype=int16)

In [11]:
# map unique CHARACTERISTIC_LEVEL count
raw_con.execute(f"""
    SELECT
        (((
         LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
        ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL,
        COUNT(*) AS n
    FROM 
        raw_da_c2021
    WHERE 
        CHARACTERISTIC_ID BETWEEN 1698 AND 1948
    GROUP BY
        CHARACTERISTIC_LEVEL
    ORDER BY
        CHARACTERISTIC_LEVEL
""").fetch_df()

,CHARACTERISTIC_LEVEL,n
0,1,13130
1,2,3282500


It is now necessary to filter `CHARACTERISTIC_LEVEL == 2` as it represents the ethnic origins without aggregation groups.

In [12]:
# new dataframe with CHARACTERISTIC LEVEL 2 only
raw_con.execute(f"""
    WITH characteristics AS (
        SELECT
            * , 
            (((
                LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
            ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
        FROM 
            raw_da_c2021
        WHERE 
            CHARACTERISTIC_ID BETWEEN 1698 AND 1948
        ORDER BY
            DGUID, CHARACTERISTIC_ID
    )

    SELECT 
        * EXCLUDE (CHARACTERISTIC_LEVEL)
    FROM
        characteristics
    WHERE
        CHARACTERISTIC_LEVEL = 2
    ORDER BY DGUID, CHARACTERISTIC_ID    
""").fetch_df()

,DGUID,CHARACTERISTIC_ID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL,C2_COUNT_MEN+,SYMBOL_1,C3_COUNT_WOMEN+,SYMBOL_2
0,2021S051235140110,1699,Canadian,105.0,NaN,60.0,NaN,45.0,NaN
1,2021S051235140110,1700,English,150.0,NaN,75.0,NaN,75.0,NaN
2,2021S051235140110,1701,Irish,170.0,NaN,80.0,NaN,95.0,NaN
3,2021S051235140110,1702,Scottish,135.0,NaN,70.0,NaN,60.0,NaN
4,2021S051235140110,1703,"French, n.o.s.",25.0,NaN,10.0,NaN,20.0,NaN
...,...,...,...,...,...,...,...,...,...
3282495,2021S051235431437,1944,Karen,0.0,NaN,0.0,NaN,0.0,NaN
3282496,2021S051235431437,1945,Edo,0.0,NaN,0.0,NaN,0.0,NaN
3282497,2021S051235431437,1946,Tajik,0.0,NaN,0.0,NaN,0.0,NaN
3282498,2021S051235431437,1947,Amhara,0.0,NaN,0.0,NaN,0.0,NaN


### Characteristic Level 2

### Pivot to wide format

The dataset will be transformed in a wide format, so that there will be one row per `DGUID`. As the dataframe has three count values: total, men+ and women+, the df will be divided by each one of the counts. This operation will keep the df readable.

In [14]:
# new CHARACTERISTIC LEVEL 2, total count only dataframe
raw_con.execute(f"""
    WITH characteristics AS (
        SELECT
            * , 
            (((
                LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
            ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
        FROM 
            raw_da_c2021
        WHERE 
            CHARACTERISTIC_ID BETWEEN 1698 AND 1948
        ORDER BY
            DGUID, CHARACTERISTIC_ID
    )

    SELECT 
        DGUID, CHARACTERISTIC_NAME, C1_COUNT_TOTAL, SYMBOL
    FROM
        characteristics
    WHERE
        CHARACTERISTIC_LEVEL = 2
    ORDER BY DGUID, CHARACTERISTIC_ID    
""").fetch_df()

,DGUID,CHARACTERISTIC_NAME,C1_COUNT_TOTAL,SYMBOL
0,2021S051235140110,Canadian,105.0,NaN
1,2021S051235140110,English,150.0,NaN
2,2021S051235140110,Irish,170.0,NaN
3,2021S051235140110,Scottish,135.0,NaN
4,2021S051235140110,"French, n.o.s.",25.0,NaN
...,...,...,...,...
3282495,2021S051235431437,Karen,0.0,NaN
3282496,2021S051235431437,Edo,0.0,NaN
3282497,2021S051235431437,Tajik,0.0,NaN
3282498,2021S051235431437,Amhara,0.0,NaN


In [15]:
# unique symbol values count
raw_con.execute(f"""
    WITH 
    characteristics AS (
        SELECT
            * , 
            (((
                LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
            ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
        FROM 
            raw_da_c2021
        WHERE 
            CHARACTERISTIC_ID BETWEEN 1698 AND 1948
        ORDER BY
            DGUID, CHARACTERISTIC_ID
    ),

    country AS (
        SELECT 
            DGUID, CHARACTERISTIC_NAME, C1_COUNT_TOTAL, SYMBOL
        FROM
            characteristics
        WHERE
            CHARACTERISTIC_LEVEL = 2
        ORDER BY DGUID, CHARACTERISTIC_ID    
    ),

    non_symbol_na AS (
        SELECT
            COALESCE(SYMBOL,'NULL') AS SYMBOL
        FROM 
            country
    )

    SELECT
        SYMBOL,
        COUNT(SYMBOL) AS n
    FROM 
        non_symbol_na
    GROUP BY 
        SYMBOL
""").fetch_df()

,SYMBOL,n
0,x,19500
1,NULL,3263000


There is no relevant quality value in `SYMBOL`. So, it can be dropped.

In [ ]:
# get all unique Characteristic Name
columns = raw_con.execute(f"""
    WITH 
        characteristics AS (
            SELECT
                * , 
                (((
                    LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
                ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
            FROM 
                raw_da_c2021
            WHERE 
                CHARACTERISTIC_ID BETWEEN 1698 AND 1948
            ORDER BY
                DGUID, CHARACTERISTIC_ID
        )
    
    SELECT DISTINCT
        CHARACTERISTIC_NAME
    FROM
        characteristics
    WHERE
        CHARACTERISTIC_LEVEL = 2
    ORDER BY CHARACTERISTIC_ID
""").fetchall()

In [17]:
pivot_columns = ", ".join(
    "'" + name.replace("'", "''") + "'"
    for (name,) in columns
)

pivot_columns

"'  Canadian', '  English', '  Irish', '  Scottish', '  French, n.o.s.', '  German', '  Chinese', '  Italian', '  Indian (India)', '  Ukrainian', '  Dutch', '  Polish', '  Québécois', '  British Isles, n.o.s.', '  Filipino', '  French Canadian', '  Caucasian (White), n.o.s.', '  First Nations (North American Indian), n.o.s.', '  Métis', '  European, n.o.s.', '  Russian', '  Norwegian', '  Welsh', '  Portuguese', '  American', '  Spanish', '  Swedish', '  Hungarian', '  Acadian', '  Pakistani', '  African, n.o.s.', '  Jewish', '  Punjabi', '  Vietnamese', '  Arab, n.o.s.', '  Greek', '  Jamaican', '  Asian, n.o.s.', '  Cree, n.o.s.', '  Korean', '  Romanian', '  Lebanese', '  Iranian', '  Christian, n.i.e.', '  Danish', '  North American Indigenous, n.o.s.', '  Sikh', '  Austrian', '  Belgian', '  Haitian', '  Hindu', '  Mexican', '  Mennonite', '  Swiss', '  Finnish', '  Sri Lankan', '  Croatian', '  Japanese', '  South Asian, n.o.s.', '  Mi''kmaq, n.o.s.', '  Northern European, n.o.s.

In [ ]:
# wide format df
raw_con.execute(f"""
    WITH 
    characteristics AS (
        SELECT
            * , 
            (((
                LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
            ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
        FROM 
            raw_da_c2021
        WHERE 
            CHARACTERISTIC_ID BETWEEN 1698 AND 1948
        ORDER BY
            DGUID, CHARACTERISTIC_ID
    ),

    country AS (
        SELECT 
            DGUID, 
            CHARACTERISTIC_NAME, C1_COUNT_TOTAL,
            CHARACTERISTIC_ID
        FROM
            characteristics
        WHERE
            CHARACTERISTIC_LEVEL = 2
        ORDER BY DGUID, CHARACTERISTIC_ID
    )

    
    PIVOT 
        country
    ON
        CHARACTERISTIC_NAME IN ({pivot_columns})
    USING
        SUM(C1_COUNT_TOTAL)
    GROUP BY
        DGUID
""").fetch_df()

,DGUID,Canadian,English,Irish,Scottish,"French, n.o.s.",German,Chinese,Italian,Indian (India),...,Costa Rican,African Nova Scotian,Malagasy,Kashmiri,Singaporean,Karen,Edo,Tajik,Amhara,Paraguayan
0,2021S051235203848,65.0,145.0,115.0,135.0,50.0,40.0,30.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021S051235203857,125.0,110.0,130.0,120.0,40.0,50.0,70.0,35.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021S051235203930,45.0,25.0,40.0,15.0,0.0,0.0,785.0,0.0,30.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021S051235203951,65.0,30.0,10.0,20.0,10.0,0.0,445.0,30.0,120.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021S051235203978,215.0,100.0,65.0,70.0,70.0,50.0,110.0,130.0,75.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13125,2021S051235431280,40.0,160.0,125.0,130.0,15.0,15.0,0.0,20.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13126,2021S051235431282,180.0,170.0,105.0,85.0,40.0,50.0,0.0,40.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13127,2021S051235431312,85.0,225.0,125.0,125.0,95.0,60.0,0.0,40.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13128,2021S051235431330,80.0,100.0,115.0,55.0,105.0,35.0,0.0,15.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# inspect NAs in wide format df
raw_con.execute(f"""
    WITH 
    characteristics AS (
        SELECT
            * , 
            (((
                LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
            ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
        FROM 
            raw_da_c2021
        WHERE 
            CHARACTERISTIC_ID BETWEEN 1698 AND 1948
        ORDER BY
            DGUID, CHARACTERISTIC_ID
    ),

    country AS (
        SELECT 
            DGUID, 
            CHARACTERISTIC_NAME, C1_COUNT_TOTAL,
            CHARACTERISTIC_ID
        FROM
            characteristics
        WHERE
            CHARACTERISTIC_LEVEL = 2
        ORDER BY DGUID, CHARACTERISTIC_ID
    )

    
    PIVOT 
        country
    ON
        CHARACTERISTIC_NAME IN ({pivot_columns})
    USING
        SUM(C1_COUNT_TOTAL)
    GROUP BY
        DGUID

""").fetch_df().isna().sum()

DGUID            0
  Canadian      78
  English       78
  Irish         78
  Scottish      78
                ..
  Karen         78
  Edo           78
  Tajik         78
  Amhara        78
  Paraguayan    78
Length: 251, dtype: int64

The dataframe has a wider format. There are consistently **78** NULL values across countries of origin. Probably in the same `DGUID`s. For the sake of practicity these values will be replaces with zero.

In [55]:
# replace NAs in wide format df with zero
raw_con.execute(f"""
    WITH 
    characteristics AS (
        SELECT
            * , 
            (((
                LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
            ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
        FROM 
            raw_da_c2021
        WHERE 
            CHARACTERISTIC_ID BETWEEN 1698 AND 1948
        ORDER BY
            DGUID, CHARACTERISTIC_ID
    ),

    country AS (
        SELECT 
            DGUID, 
            CHARACTERISTIC_NAME, C1_COUNT_TOTAL,
            CHARACTERISTIC_ID
        FROM
            characteristics
        WHERE
            CHARACTERISTIC_LEVEL = 2
        ORDER BY DGUID, CHARACTERISTIC_ID
    )

    
    PIVOT 
        country
    ON
        CHARACTERISTIC_NAME IN ({pivot_columns})
    USING
        SUM(COALESCE(C1_COUNT_TOTAL,0))
    GROUP BY
        DGUID

""").fetch_df()

,DGUID,Canadian,English,Irish,Scottish,"French, n.o.s.",German,Chinese,Italian,Indian (India),...,Costa Rican,African Nova Scotian,Malagasy,Kashmiri,Singaporean,Karen,Edo,Tajik,Amhara,Paraguayan
0,2021S051235191151,90.0,115.0,75.0,65.0,30.0,50.0,25.0,35.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021S051235191154,65.0,90.0,20.0,25.0,30.0,40.0,15.0,125.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021S051235191158,55.0,55.0,55.0,20.0,10.0,35.0,0.0,95.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021S051235191169,85.0,60.0,40.0,45.0,0.0,15.0,645.0,25.0,290.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021S051235191205,35.0,40.0,45.0,55.0,35.0,0.0,225.0,30.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13125,2021S051235280325,130.0,190.0,90.0,145.0,25.0,50.0,0.0,10.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13126,2021S051235280330,100.0,210.0,125.0,130.0,25.0,65.0,0.0,10.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13127,2021S051235280337,50.0,105.0,50.0,75.0,0.0,55.0,0.0,25.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13128,2021S051235280398,45.0,170.0,75.0,125.0,45.0,40.0,0.0,15.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


The wide table will be stored separately in new database dedicated to transformed data. The wide table will be created in the raw database then copied to the new one.

In [59]:
# create the wide format df
raw_con.execute(f"""
    
    CREATE OR REPLACE TABLE wide_total_da_c2021 AS
    
    WITH 
    characteristics AS (
        SELECT
            * , 
            (((
                LENGTH(CHARACTERISTIC_NAME) - LENGTH(LTRIM(CHARACTERISTIC_NAME)) 
            ) / 2) + 1):: SMALLINT AS CHARACTERISTIC_LEVEL
        FROM 
            raw_da_c2021
        WHERE 
            CHARACTERISTIC_ID BETWEEN 1698 AND 1948
        ORDER BY
            DGUID, CHARACTERISTIC_ID
    ),

    country AS (
        SELECT 
            DGUID, 
            CHARACTERISTIC_NAME, C1_COUNT_TOTAL,
            CHARACTERISTIC_ID
        FROM
            characteristics
        WHERE
            CHARACTERISTIC_LEVEL = 2
        ORDER BY DGUID, CHARACTERISTIC_ID
    )

    
    PIVOT 
        country
    ON
        CHARACTERISTIC_NAME IN ({pivot_columns})
    USING
        SUM(COALESCE(C1_COUNT_TOTAL,0))
    GROUP BY
        DGUID

""")

In [5]:
# define path variables
raw = 'data/database/raw.duckdb'
silver ='data/database/silver.duckdb'

In [6]:
# making sure that silver file is new
if os.path.exists(silver):
    os.remove(silver)

In [8]:
raw_con = duckdb.connect("data/database/raw.duckdb")

In [ ]:
# copy the wide table inot a fresh database
raw_con.execute(f"""
    ATTACH 
        'data/database/silver.duckdb' AS silver;
    CREATE 
        TABLE silver.wide_total_da_c2021 AS
    SELECT
        *
    FROM raw.wide_total_da_c2021;
    
    DETACH 
        silver;
""")

In [11]:
# close connection
raw_con.close()

In [12]:
# Create a DuckDB database
silver_con = duckdb.connect("data/database/silver.duckdb")

In [13]:
silver_con.execute(f"""

    SELECT *
    FROM wide_total_da_c2021

""").fetch_df()

,DGUID,Canadian,English,Irish,Scottish,"French, n.o.s.",German,Chinese,Italian,Indian (India),...,Costa Rican,African Nova Scotian,Malagasy,Kashmiri,Singaporean,Karen,Edo,Tajik,Amhara,Paraguayan
0,2021S051235200380,20.0,30.0,45.0,25.0,0.0,40.0,220.0,20.0,30.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2021S051235200385,50.0,25.0,0.0,10.0,0.0,0.0,90.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2021S051235200422,45.0,20.0,35.0,0.0,0.0,0.0,30.0,20.0,90.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2021S051235200437,20.0,20.0,0.0,30.0,0.0,0.0,35.0,35.0,50.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2021S051235200463,20.0,80.0,45.0,15.0,20.0,25.0,0.0,0.0,50.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13125,2021S051235430937,165.0,165.0,155.0,110.0,60.0,95.0,0.0,50.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13126,2021S051235430944,60.0,95.0,90.0,80.0,25.0,65.0,0.0,50.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13127,2021S051235430950,270.0,460.0,205.0,210.0,100.0,190.0,30.0,150.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13128,2021S051235430978,30.0,140.0,55.0,105.0,25.0,65.0,0.0,115.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# close connection
silver_con.close()

# Next Step

Explore the `Dissemination Areas Boundaries` to get the spatial features needed to visualize population data

# Resouces

- [Census Profile, 2021 Census of Population](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/index.cfm?Lang=E)
- [About the Census Profile, 2021 Census of Population](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/about-apropos/about-apropos.cfm?Lang=E#aa1)
- [Guide to the Census of Population, 2021](https://www12.statcan.gc.ca/census-recensement/2021/ref/98-304/index-eng.cfm),  provides an overview of the Census of Population content determination, collection, processing, data quality assessment and data dissemination. It may be useful to both new and experienced users who wish to familiarize themselves with and find specific information about the 2021 Census
- [Filling the gaps: Information on gender in the 2021 Census](https://www12.statcan.gc.ca/census-recensement/2021/ref/98-20-0001/982000012021001-eng.cfm), defines gender, sex at birth, and relevant concepts as the Census 2021 disseminates census information on gender
- [Census Profile metadata](https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/about-apropos/metadata-metadonnees-eng.cfm), list characteristics by topics and subtopic, and list all footnotes
- [Full Table Download (CSV) User Guide](https://www.statcan.gc.ca/en/developers/csv/user-guide), provides users with a guide to the full table downloadable output files available from the Statistics Canada website
- [Dictionary, Census of Population 2021, PDF version](https://www12.statcan.gc.ca/census-recensement/2021/ref/dict/98-301-x2021001-eng.pdf), is a reference document which contains detailed definitions of Census of Population concepts, variables and geographic terms, as well as historical information. The PDF version organizes the concepts by topics, which is not the case for the [web version](https://www12.statcan.gc.ca/census-recensement/2021/ref/dict/index-eng.cfm).

# Reference

Statistics Canada. 2023. Census Profile. 2021 Census of Population. Statistics Canada Catalogue number 98-316-X2021001. Ottawa. Released November 15, 2023.
*https://www12.statcan.gc.ca/census-recensement/2021/dp-pd/prof/index.cfm?Lang=E (accessed August 18, 2026).*